主要是将ml_lab2_pytorch_mnist.ipynb中关于Res34/50 for MNIST的代码摘取出来

In [ ]:
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import precision_score, recall_score, f1_score

import matplotlib.pyplot as plt


# 固定随机种子，方便复现实验结果。
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# 有 GPU 就使用 GPU，否则使用 CPU。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", device)


In [ ]:
# MNIST 是 1 通道灰度图，所以均值和标准差都只有一个数。
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)

transform = transforms.Compose([
    transforms.ToTensor(),                         # 0~255 -> 0~1
    transforms.Normalize(MNIST_MEAN, MNIST_STD),   # 标准化
])


In [ ]:
import os 
print("current working directory:",os.getcwd())
#expected:\lab2   确保读取MNIST数据集到lab2目录下

In [ ]:
# 下载并读取 MNIST 数据集。
train_full = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, download=True, transform=transform)

TRAIN_SIZE = 50000   # 用于训练
VAL_SIZE = 10000     # 用于调参与画曲线

# 使用固定随机种子划分，保证每次运行划分结果一致。
train_dataset, val_dataset = random_split(
    train_full,
    [TRAIN_SIZE, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)

print("训练集样本数:", len(train_dataset))
print("验证集样本数:", len(val_dataset))
print("测试集样本数:", len(test_dataset))


In [ ]:
#超参数设置
batch_size=64

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("每个 batch 的样本数:", batch_size)
print("训练集 batch 数量:", len(train_loader))
#TODO:可以进行数据集增强,对于MNIST的数字分类问题,可以使用旋转增强

## 相关网络层

### 卷积参数
3*3, stride=1,padding=1: 保持特征图大小不变,用于提取局部特征  
1*1,stride=1,padding=0:保持特征图大小不变,用于改变通道数(e.g. projection connect)  
3*3, stride=2, padding=1: 用于将特征图H和W各自减半,同时上升通道数  

输出尺寸的计算公式:  
H'=math.floor((H-K+2*Padding)/stride)+1  
对于3*3,stride=2,padding=1,  (H-3+2)/2=(H-1)/2  
当H为偶数时,设H=2*k,则math.floor(H-1/2)=k-1, 从而H'=k-1+1=k == H/2  
当H为奇数时,设H=2*k+1,则(H-1)/2=k,H'=k+1=(H-1)/2+1  
由于MNIST的图片大小为28*28,经过Stage2,Stage3,Stage4共3次减半  
且在Stage4时,大小由7减小到4  (代入公式可得,(7-1)/2+1=4)  
因此特征图的H和W各自减半  

### nn.Conv2d()
nn.Conv2d(in_channels,out_channels,kernel_size,stride=1,padding=0,bias=True)  
由于Res34/50中Conv都紧跟着BatchNorm,因此需要设置bias=False


### nn.BatchNorm2d()
nn.BatchNorm2d(num_features, eps=1e-5, momentum=0.1, affine=True)  
由于Conv后紧跟BN,此处num_features就是卷积的输出通道数out_channels  
affine=True代表具有可学习的参数γ和β  

ResNet使用的是BatchNorm2d  
对于(N,C,H,W)的特征图,每个通道视作一个独立的特征提取器  
(  对应于全连接网络一层中的单个神经元  )  
对一个批次中的所有样本,一个通道中所有位置的像素计算统计量  
(即,给定一个通道,用H * W * C个元素计算它对应的均值与方差)  

注意,由于使用了BN,测试时使用训练时的指数移动平均统计量进行归一化  
所以训练时需要```model.train()```, ```model.eval()```

### nn.MaxPool2d()
nn.MaxPool2d(kernel_size=3,stride=2,padding=1)  
将特征图尺寸减半(H',W'=H/2,W/2)  
由于MNIST中初始图片形状为(1,28,28)  
因此不需要下采样.   使用nn.Identity()代替

### nn.CrossEntropyLoss()
self.criterion = nn.CrossEntropyLoss()  
self.criterion(logits, target)得到损失.  
设最终输出的嵌入向量为vec,形状为(N,H)  
label形状为(N,)  
nn.CrossEntropyLoss(vec,label,reduction='mean')  


### nn.AvgPool2d(kernel_size)
kernel_size在此处就类似MaxPool2d(kernel_size)中的kernel_size  
代表对kernel_size*kernel_size大小的区域聚合为一个数    
由于AvgPool对整张图片一个通道中所有元素进行聚合,  
因此这里kernel_size实际上就指这个图片的H或者W  
对于此处的Res34/50而言,最终输出图片大小为4*4  
因此kernel_size=4  

同时,类似MaxPool2d()返回的图片依旧是(N,C,H,W)形状  
nn.AvgPool2d(kernel_size)返回的图片形状为(N,C,1,1)  
因此进行线性层时,需要先flatten (i.e. torch.flatten(y,1))  
把1维度及其之后的维度合并,i.e., (C,1,1)->(C,)  

## nn.AdaptiveAvgPool2d()
nn.AvgPool2d(kernel_size=4)可以替换为  
nn.AdaptiveAvgPool2d((1,1))


## 整体架构
### 共同之处
Stem:
7*7,64,stride 2 -> BN2d -> ReLU ->  
MaxPool 3*3, stride 2, padding=1  
-> Residual Stages:  
Stage1 ->  Stage2 -> Stage3 -> Stage4     
Conv2d_x   Conv3d_x  Conv4d_x  Conv5d_x     
-> Classify_head  
GAP -> Fully Connected (1000 for ImageNet, 10 for MNIST)  

### Res34
BasicBlock: 2个卷积层  
保持特征图尺寸不变  
3*3,stride=1,padding=1  


每个Stage的第一个BasicBlock需要负责  
通道数加倍,尺寸减半  
该BasicBlock的第一个卷积核为3*3,stride=2,padding=1  
且采用Projection connection : 1*1,stride=1,padding=0,channels加倍  
**所有Projection connection**后面都必须紧跟一个BatchNorm2d  
实际上,**所有卷积层后面都要紧跟一个BatchNorm2d** (**包括Stem阶段**)  



### Res50
BottleNeck: 3个卷积层: 降维->卷积->升维 整个过程保持特征图尺寸不变  
1*1, stride=1, padding=0, out_channels = C ->  
3*3, stride=1, padding=1, out_channels = C ->  
1*1, stride=1, padding=0, out_channels = 4C  
其中4C才是该Stage的通道数量,整个Stage内特征图传递的通道数为4C  






## Res34 for MNIST

In [ ]:
class BasicBlock(nn.Module):
    """
    input: x (N,C,H,W), H==W
    两个卷积层
    常规卷积层为3*3,stride=1,padding=1
    保持通道数与尺寸不变
    残差连接:直接相加

    如果down_sampling=True
    则第一个卷积层为3*3,stride=2,padding=1
    使得通道数加倍,尺寸减半
    且残差连接使用1*1卷积;
    
    """
    def __init__(self,out_channels,down_sampling=False):
        """
        down_sampling:是否进行下采样,i.e.,通道数加倍,尺寸图减半
        out_channels:代表该Stage的channels
        如果down_sampling=True
        则in_channels=out_channels/2
        否则in_channels=out_channels
        """
        super().__init__()

        self.down_sampling=down_sampling 
        in_channels= out_channels//2 if self.down_sampling else out_channels
        stride= 2 if self.down_sampling else 1
        self.Conv1 = nn.Conv2d(in_channels,out_channels,
                               kernel_size=3,stride=stride,padding=1,
                               bias=False)
        self.BN1 = nn.BatchNorm2d(out_channels)
        self.Conv2 = nn.Conv2d(out_channels,out_channels,
                               kernel_size=3,stride=1,padding=1,
                               bias=False)
        #第二个常规卷积的stride总为1
        self.BN2 = nn.BatchNorm2d(out_channels)
        if self.down_sampling:
            self.projection = nn.Conv2d(in_channels,out_channels,
                                       kernel_size=1,stride=2,padding=0,
                                       bias=False)
            #注意,stride=2,因为此时特征图尺寸减半
            #对于偶数,返回H/2;对于奇数,返回(H+1)/2 
            self.BN3 = nn.BatchNorm2d(out_channels)

        


    def forward(self,x):
        y = F.relu(self.BN1(self.Conv1(x)))
        y = self.BN2(self.Conv2(y))
        if self.down_sampling:
            x = self.BN3(self.projection(x))
        return F.relu(y+x)
    

In [ ]:
class Res34(nn.Module):
    def __init__(self,in_channels=1):
        """
        Res34 for MNIST Classification
        input: image of shape (N,1,28,28)
        output: predict one of 10 class

        Stem: 
        3*3, stride=1,padding=1,out_channels=64
        BatchNorm2d ->ReLU
        No MaxPooling Layer

        channels: 64->128->256->512  size: 28 14 7 4
        4 Residual Stages
        #BasicBlocks: 3,4,6,3
        Stage1 <--> Conv2_x
        Stage2 <--> Conv3_x
        Stage3 <--> Conv4_x
        Stage4 <--> Conv5_x 

        Stage4 输出的张量形状为(N,512,4,4)
        """
        super().__init__()
        self.in_channels=in_channels
        self.stage1_c,self.stage1_num = 64 , 3
        self.stage2_c, self.stage2_num = 128, 4
        self.stage3_c, self.stage3_num = 256, 6
        self.stage4_c, self.stage4_num = 512, 3
    
        self.stem = nn.Sequential(*[nn.Conv2d(self.in_channels,self.stage1_c,
                                kernel_size=3,stride=1,padding=1,
                                bias=False ),
                                 nn.BatchNorm2d(self.stage1_c),
                                  nn.ReLU() ])
        #由于使用MNIST,Stage1不需要进行down_sampling(原本需要经过max_pooling)
        self.stage1 = nn.Sequential(*[BasicBlock(self.stage1_c)
                                     for i in range(self.stage1_num)
        ])

        self.stage2= nn.Sequential(*([BasicBlock(self.stage2_c,down_sampling=True)]+
                                     [BasicBlock(self.stage2_c) for i in range(self.stage2_num-1)]))
        self.stage3 = nn.Sequential(*([BasicBlock(self.stage3_c,down_sampling=True)]+
                                     [BasicBlock(self.stage3_c) for i in range(self.stage3_num-1)]))
        self.stage4 =  nn.Sequential(*([BasicBlock(self.stage4_c,down_sampling=True)]+
                                     [BasicBlock(self.stage4_c) for i in range(self.stage4_num-1)]))
        """
        alternative:记录self.stage_num,把self.stage{i}_c,self.stage{i}_num;
        self.stage{i}分别封装为列表
        """
        self.stages=nn.Sequential(self.stage1,self.stage2,self.stage3,self.stage4)
        
        #最终得到特征图形状为(N,512,4,4)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))  #alternative : nn.AvgPool2d (kernel_size=4)
        self.fc = nn.Linear(self.stage4_c,10) #10分类问题


    def forward(self,x):
        y = self.stem(x)
        y = self.stages(y)
        y = self.avgpool(y)
        y = torch.flatten(y,1)  #注意展平,avgpool返回(N,C,1,1)
        y = self.fc(y)
        return y

        
        

## Res50 for MNIST

In [ ]:
class BottleNeck(nn.Module):
    def __init__(self,out_channels,down_sampling=False,first_stage=False):
        """
        降维:1*1,stride=1,padding=0,in_channels=out_channels/4 if down_sampling 
        卷积:3*3,stride=2 或 1,padding=1
        升维:1*1,stride=1,padding=0
        channels:
        out_channels/4->out_channels/4->out_channels
        
        Res50: out_channels= 256, 512, 1024, 2048

        由于经过Res50的Stem后图片的通道数为64,
        而Stage1的通道数为256,
        实际上输入通道数不满足输出通道数除以2;
        此时in_channels= out_channels/4
        因此first_stage标志用于特殊处理Stage1的情况
        """
        super().__init__()
        #保存forward()中需要使用的变量
        self.down_sampling=down_sampling
        self.first_stage=first_stage 

        in_channels = out_channels//4 if first_stage else out_channels//2 if self.down_sampling else out_channels
                            
        stride=1 if first_stage else 2 if self.down_sampling else 1
        #如果是Stage1,那么不进行下采样

        self.Conv1= nn.Conv2d(in_channels,out_channels//4,
                            kernel_size=1,stride=1,padding=0)
        self.BN1 = nn.BatchNorm2d(out_channels//4)

        self.Conv2 = nn.Conv2d(out_channels//4,out_channels//4,
                               kernel_size=3,stride=stride,padding=1)
        #由中间卷积层负责下采样(如果进行下采样的话)
        self.BN2 = nn.BatchNorm2d(out_channels//4)

        self.Conv3 = nn.Conv2d(out_channels//4,out_channels,
                                kernel_size=1,stride=1,padding=0)
        self.BN3 = nn.BatchNorm2d(out_channels)
        
        
        """
        无论下采样还是first_stage的第一层,都需要投影
        对于下采样,输入输出的尺寸与通道数都不同
        对于first_stage的第一层,输入输出的尺寸相同
        但通道数不同
        """
        if self.down_sampling or self.first_stage:
            self.projection = nn.Conv2d(in_channels,out_channels,
                                        kernel_size=1,stride=stride,padding=0)
            #下采样时:stride=2 
            self.BN4=nn.BatchNorm2d(out_channels)

        


    def forward(self,x):
        y = F.relu(self.BN1(self.Conv1(x)))
        y = F.relu(self.BN2(self.Conv2(y)))
        y = self.BN3(self.Conv3(y))
        if self.down_sampling or self.first_stage:
            x = self.BN4(self.projection(x))
        return F.relu(y+x)
    

In [ ]:
class Res50(nn.Module):
    """
    Res50 for MNIST
    channels: 256, 512, 1024, 2048
    size:28 -> 14 -> 7 ->4

    Stem:
    3*3,stride=1,padding=1,out_channels=64
    No MaxPooling

    Stage1->Stage2->Stage3->Stage4->GAP+Linear

    Stage1: C_in:64 C_out:256
    Stage1的第一个BottleNeck的第一层卷积的输入通道需要特殊处理
    此时C_in≠C_out//2,而是C_out//4

    """
    def __init__(self,in_channels=1):
        super().__init__()
        self.in_channels=in_channels
        self.stem_c=64
        self.stage1_c,self.stage1_num=256,3
        self.stage2_c,self.stage2_num=512,4
        self.stage3_c,self.stage3_num=1024,6
        self.stage4_c,self.stage4_num=2048,3

        self.stem= nn.Sequential(*[nn.Conv2d(in_channels,self.stem_c,
                                            kernel_size=3,stride=1,padding=1,
                                            bias=False),
                                    nn.BatchNorm2d(self.stem_c),
                                    nn.ReLU()
                             ])
        self.stage1=nn.Sequential(*([BottleNeck(self.stage1_c,first_stage=True)]+
                                    [BottleNeck(self.stage1_c) for i in range(self.stage1_num -1)]))
        self.stage2=nn.Sequential(*([BottleNeck(self.stage2_c,down_sampling=True)]+
                                    [BottleNeck(self.stage2_c) for i in range(self.stage2_num-1)]))
        self.stage3=nn.Sequential(*([BottleNeck(self.stage3_c,down_sampling=True)]+
                                    [BottleNeck(self.stage3_c) for i in range(self.stage3_num-1)]))
        self.stage4=nn.Sequential(*([BottleNeck(self.stage4_c,down_sampling=True)]+
                                    [BottleNeck(self.stage4_c) for i in range(self.stage4_num-1)]))
        self.stages=nn.Sequential(self.stage1,self.stage2,self.stage3,self.stage4)
        
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.fc=nn.Linear(self.stage4_c,10)
        
    def forward(self,x):
        y =self.stem(x)
        y = self.stages(y)
        y = self.avgpool(y)
        y = torch.flatten(y,1)
        y = self.fc(y)
        return y

    

In [ ]:
#sanity check 
model34= Res34()
model50=Res50()
x=torch.randn(2,1,28,28)

print(model34(x).shape)
print(model50(x).shape)